<a href="https://colab.research.google.com/github/k9Sx3CC/01_first_look_and_discovery.ipynb/blob/main/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k9Sx3CC/flyrank-internship-test/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [3]:
%pip -q install duckdb huggingface_hub


In [4]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [5]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [6]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [7]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [8]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,1.0,0.022750,0.957216,59.0,59.0,1.000000
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,14.0,0.017946,0.932994,84.0,462.0,0.181818
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,3.0,0.162037,0.552469,153.0,185.0,0.827027
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,2.0,0.108932,0.820261,52.0,65.0,0.800000
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,5.0,0.163052,0.788332,14.0,65.0,0.215385


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [9]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.546     0.332     0.413      9389
           1      0.684     0.840     0.754     16162

    accuracy                          0.653     25551
   macro avg      0.615     0.586     0.583     25551
weighted avg      0.633     0.653     0.629     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [10]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

# ============================================================================
# STEP 1: Build features with 90-day window and position volatility
# ============================================================================
print("="*60)
print("STEP 1: Building 90-day features with position volatility")
print("="*60)

features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    daily_data AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            EXTRACT(DOW FROM report_date) AS day_of_week
        FROM {TABLES['fact_daily']}
        WHERE report_date > (SELECT end_d - INTERVAL 90 DAY FROM bounds)
    ),
    aggregated AS (
        SELECT
            client_hash_id,
            content_hash_id,
            -- 45-day split (90-day window = 45/45 split)
            SUM(CASE WHEN report_date > (SELECT end_d - INTERVAL 45 DAY FROM bounds)
                THEN gsc_impressions ELSE 0 END) AS imp_last45,
            SUM(CASE WHEN report_date <= (SELECT end_d - INTERVAL 45 DAY FROM bounds)
                THEN gsc_impressions ELSE 0 END) AS imp_prev45,
            -- Clicks
            SUM(CASE WHEN report_date > (SELECT end_d - INTERVAL 45 DAY FROM bounds)
                THEN gsc_clicks ELSE 0 END) AS clk_last45,
            -- Position metrics (NEW FEATURE: volatility)
            AVG(CASE WHEN report_date > (SELECT end_d - INTERVAL 45 DAY FROM bounds)
                THEN gsc_avg_position END) AS pos_avg_last45,
            STDDEV(gsc_avg_position) AS pos_volatility,
            -- Weekend share (NEW FEATURE)
            SUM(CASE WHEN day_of_week IN (0, 6) THEN gsc_impressions ELSE 0 END) * 1.0 /
                NULLIF(SUM(gsc_impressions), 0) AS weekend_share,
            -- CTR metrics
            SUM(CASE WHEN report_date > (SELECT end_d - INTERVAL 45 DAY FROM bounds)
                THEN gsc_clicks ELSE 0 END) * 1.0 /
                NULLIF(SUM(CASE WHEN report_date > (SELECT end_d - INTERVAL 45 DAY FROM bounds)
                    THEN gsc_impressions ELSE 0 END), 0) AS ctr_last45
        FROM daily_data
        GROUP BY client_hash_id, content_hash_id
        -- HAVING threshold: require at least 500 impressions in prev period
        HAVING SUM(CASE WHEN report_date <= (SELECT end_d - INTERVAL 45 DAY FROM bounds)
                    THEN gsc_impressions ELSE 0 END) >= 500
    )
    SELECT * FROM aggregated
""").df()

print(f"✓ {len(features_90d):,} content items with 90-day history (threshold: 500 prev impressions)")

# ============================================================================
# STEP 2: Add query-level signals
# ============================================================================
print("\n" + "="*60)
print("STEP 2: Adding query-level signals")
print("="*60)

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']

data = features_90d.merge(qsignals, on='content_hash_id', how='left')
print(f"✓ Joined: {len(data):,} rows")

# ============================================================================
# STEP 3: Define label and prepare features
# ============================================================================
print("\n" + "="*60)
print("STEP 3: Defining label and preparing features")
print("="*60)

# Label: impressions decline by >20% from prev45 to last45
data['is_declining'] = (data['imp_last45'] < 0.8 * data['imp_prev45']).astype(int)

# Feature set (including our two new features)
feature_cols = [
    'imp_prev45',           # Baseline traffic
    'visible_queries',      # Query diversity
    'top_query_share',      # Concentration
    'rare_share',           # Tail distribution
    'anon_share',           # Anonymized traffic
    'pos_volatility',       # NEW: Position stability (lower = more stable)
    'weekend_share',        # NEW: Weekend traffic pattern
]

# Prepare data
model_data = data.dropna(subset=feature_cols + ['client_hash_id'])
X = model_data[feature_cols].values
y = model_data['is_declining'].values
groups = model_data['client_hash_id'].values

print(f"✓ Total samples: {len(y):,}")
print(f"✓ Declining rate: {y.mean():.3f}")
print(f"✓ Unique clients: {len(np.unique(groups)):,}")

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============================================================================
# STEP 4: Compare Random Split vs Client-Aware Split
# ============================================================================
print("\n" + "="*60)
print("STEP 4: Comparing Random Split vs GroupShuffleSplit")
print("="*60)

# Random split
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

# GroupShuffleSplit (client-aware)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X_scaled, y, groups))
X_tr_group, X_te_group = X_scaled[train_idx], X_scaled[test_idx]
y_tr_group, y_te_group = y[train_idx], y[test_idx]
groups_te = groups[test_idx]

print(f"\n{'Random Split':30} | {'GroupShuffleSplit':30}")
print("-"*61)
print(f"Train samples: {len(y_tr_rand):,}       | Train samples: {len(y_tr_group):,}")
print(f"Test samples:  {len(y_te_rand):,}       | Test samples:  {len(y_te_group):,}")
print(f"Train clients: {len(np.unique(groups[train_idx])):,}      | Train clients: {len(np.unique(groups[train_idx])):,}")
print(f"Test clients:  {len(np.unique(groups[test_idx])):,}      | Test clients:  {len(np.unique(groups[test_idx])):,}")

# ============================================================================
# STEP 5: Train and evaluate models
# ============================================================================
print("\n" + "="*60)
print("STEP 5: Model Training and Evaluation")
print("="*60)

def evaluate_model(X_tr, X_te, y_tr, y_te, split_name, groups_te=None):
    """Train RandomForest and return metrics"""
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5
    )
    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    return {
        'model': model,
        'accuracy': (y_pred == y_te).mean(),
        'auc_roc': roc_auc_score(y_te, y_proba),
        'f1': classification_report(y_te, y_pred, output_dict=True)['weighted avg']['f1-score'],
        'y_pred': y_pred,
        'y_proba': y_proba
    }

# Evaluate both splits
results_rand = evaluate_model(X_tr_rand, X_te_rand, y_tr_rand, y_te_rand, "Random")
results_group = evaluate_model(X_tr_group, X_te_group, y_tr_group, y_te_group, "Group", groups_te)

# Performance comparison
print(f"\n{'Metric':20} | {'Random Split':15} | {'GroupShuffleSplit':15} | {'Delta':10}")
print("-"*62)
print(f"{'Accuracy':20} | {results_rand['accuracy']:.4f}        | {results_group['accuracy']:.4f}        | {results_group['accuracy']-results_rand['accuracy']:+.4f}")
print(f"{'AUC-ROC':20} | {results_rand['auc_roc']:.4f}        | {results_group['auc_roc']:.4f}        | {results_group['auc_roc']-results_rand['auc_roc']:+.4f}")
print(f"{'F1 Score':20} | {results_rand['f1']:.4f}        | {results_group['f1']:.4f}        | {results_group['f1']-results_rand['f1']:+.4f}")

# ============================================================================
# STEP 6: Feature Importance Analysis
# ============================================================================
print("\n" + "="*60)
print("STEP 6: Feature Importance (GroupShuffleSplit Model)")
print("="*60)

# Get feature importance from group model
importances = results_group['model'].feature_importances_
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

print(f"\nFeature Importance:")
print(feature_importance.to_string(index=False))

# Which features carry the signal?
print("\n✓ Top signals for predicting decline:")
for i, row in feature_importance.head(3).iterrows():
    print(f"  • {row['feature']}: {row['importance']:.3f}")

# ============================================================================
# STEP 7: Performance by Client Size (Diagnostic)
# ============================================================================
print("\n" + "="*60)
print("STEP 7: Performance by Client Size")
print("="*60)

def evaluate_by_client_size(y_true, y_pred, groups_te, n_bins=4):
    """Evaluate performance segmented by client size"""
    client_sizes = pd.Series(groups_te).value_counts()
    size_groups = pd.qcut(client_sizes, q=n_bins, labels=[f'Size Q{i+1}' for i in range(n_bins)])

    results = []
    for group_name in size_groups.cat.categories:
        clients_in_group = client_sizes[size_groups == group_name].index
        mask = np.isin(groups_te, clients_in_group)

        if mask.sum() > 0:
            acc = (y_true[mask] == y_pred[mask]).mean()
            auc = roc_auc_score(y_true[mask], y_pred[mask]) if len(np.unique(y_true[mask])) > 1 else 0.5
            results.append({
                'client_size_group': group_name,
                'n_samples': mask.sum(),
                'n_clients': len(clients_in_group),
                'accuracy': acc,
                'auc': auc,
                'base_rate': y_true[mask].mean()
            })

    return pd.DataFrame(results)

# Add predictions to evaluate
y_pred_group = results_group['y_pred']
results_by_size = evaluate_by_client_size(y_te_group, y_pred_group, groups_te)

print("\nModel Performance by Client Size:")
print(results_by_size.to_string(index=False))

# ============================================================================
# STEP 8: Confusion Matrix
# ============================================================================
print("\n" + "="*60)
print("STEP 8: Confusion Matrix (GroupShuffleSplit)")
print("="*60)

cm = confusion_matrix(y_te_group, y_pred_group)
print(f"\n                 Predicted")
print(f"                 No Decline  Decline")
print(f"Actual No Decline    {cm[0,0]:>6,}   {cm[0,1]:>6,}")
print(f"       Decline       {cm[1,0]:>6,}   {cm[1,1]:>6,}")

# ============================================================================
# STEP 9: Summary and Conclusion
# ============================================================================
print("\n" + "="*60)
print("SUMMARY: Does the signal generalize across clients?")
print("="*60)

# Interpret results
auc_drop = results_rand['auc_roc'] - results_group['auc_roc']
if results_group['auc_roc'] > 0.7:
    conclusion = "✓ STRONG: Model generalizes well across clients"
elif results_group['auc_roc'] > 0.6:
    conclusion = "→ MODERATE: Some generalization, but room for improvement"
else:
    conclusion = "✗ WEAK: Model doesn't generalize - likely overfitting to client patterns"

print(f"\n{conclusion}")
print(f"Random AUC: {results_rand['auc_roc']:.3f} → Group AUC: {results_group['auc_roc']:.3f} (drop: {auc_drop:.3f})")

print("\nKey Takeaway:")
if feature_importance.iloc[0]['feature'] == 'pos_volatility':
    print("  • Position volatility is the strongest predictor - pages with unstable rankings tend to decline")
elif feature_importance.iloc[0]['feature'] == 'weekend_share':
    print("  • Weekend share patterns predict decline - suggests business vs. informational content")
else:
    print(f"  • {feature_importance.iloc[0]['feature']} is the primary driver")

print("\nCapstone Grade Assessment:")
if results_group['auc_roc'] > 0.65 and auc_drop < 0.1:
    print("  ⭐ EXCELLENT: Model generalizes and feature engineering is robust")
elif results_group['auc_roc'] > 0.6:
    print("  ★ GOOD: Interesting signal, but consider additional features")
else:
    print("  → NEEDS WORK: Try different features or more granular temporal patterns")

# Optional: Save your feature table
data.to_parquet('my_90d_features.parquet', index=False)
print("\n✓ Saved feature table to 'my_90d_features.parquet'")

STEP 1: Building 90-day features with position volatility


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ 73,860 content items with 90-day history (threshold: 500 prev impressions)

STEP 2: Adding query-level signals
✓ Joined: 73,860 rows

STEP 3: Defining label and preparing features
✓ Total samples: 71,617
✓ Declining rate: 0.664
✓ Unique clients: 42

STEP 4: Comparing Random Split vs GroupShuffleSplit

Random Split                   | GroupShuffleSplit             
-------------------------------------------------------------
Train samples: 53,712       | Train samples: 67,720
Test samples:  17,905       | Test samples:  3,897
Train clients: 31      | Train clients: 31
Test clients:  11      | Test clients:  11

STEP 5: Model Training and Evaluation

Metric               | Random Split    | GroupShuffleSplit | Delta     
--------------------------------------------------------------
Accuracy             | 0.7616        | 0.7419        | -0.0197
AUC-ROC              | 0.8155        | 0.7923        | -0.0233
F1 Score             | 0.7464        | 0.7258        | -0.0206

STEP 6: Feature

In [12]:
# ============================================================================
# DIAGNOSTIC: Check GroupShuffleSplit Balance
# ============================================================================
print("\n" + "="*60)
print("DIAGNOSTIC: Checking GroupShuffleSplit Balance")
print("="*60)

# Check distribution of samples per client in test set
test_client_counts = pd.Series(groups_te).value_counts()
print(f"\nTest Set Client Distribution:")
print(f"Total clients in test: {len(test_client_counts)}")
print(f"Average samples per client: {test_client_counts.mean():.1f}")
print(f"Median samples per client: {test_client_counts.median():.1f}")
print(f"Min samples per client: {test_client_counts.min()}")
print(f"Max samples per client: {test_client_counts.max()}")

print(f"\nClient Size Breakdown:")
print(f"  Clients with < 10 samples: {(test_client_counts < 10).sum()}")
print(f"  Clients with 10-100 samples: {((test_client_counts >= 10) & (test_client_counts < 100)).sum()}")
print(f"  Clients with 100-1000 samples: {((test_client_counts >= 100) & (test_client_counts < 1000)).sum()}")
print(f"  Clients with > 1000 samples: {(test_client_counts >= 1000).sum()}")

# ============================================================================
# DIAGNOSTIC: Client Size vs Performance
# ============================================================================
print("\n" + "="*60)
print("DIAGNOSTIC: Client Size vs Performance")
print("="*60)

client_perf = []
for client_id in np.unique(groups_te):
    mask = groups_te == client_id
    if mask.sum() > 0:
        y_true_client = y_te_group[mask]
        y_pred_client = results_group['y_pred'][mask]

        client_perf.append({
            'client_id': client_id,
            'n_samples': mask.sum(),
            'accuracy': (y_true_client == y_pred_client).mean(),
            'auc': roc_auc_score(y_true_client, y_pred_client) if len(np.unique(y_true_client)) > 1 else 0.5,
            'base_rate': y_true_client.mean()
        })

perf_df = pd.DataFrame(client_perf)
perf_df['size_group'] = pd.cut(perf_df['n_samples'],
                               bins=[0, 10, 100, 1000, float('inf')],
                               labels=['Tiny (<10)', 'Small (10-100)', 'Medium (100-1000)', 'Large (>1000)'])

print(f"\nPerformance by Client Size Group:")
print(perf_df.groupby('size_group').agg({
    'n_samples': ['count', 'sum', 'mean'],
    'auc': ['mean', 'std'],
    'accuracy': ['mean', 'std']
}).round(4))
print("The GroupShuffleSplit strategy ensured that no client appeared in both the training and testing sets, preventing client-level information leakage. However, client sizes were highly imbalanced, with the median test client containing 29 samples compared with a mean of 354 samples. Two large clients accounted for approximately 87% of the test observations, indicating that sample-level evaluation metrics were primarily influenced by these clients. To better understand model behavior across different client populations, performance was also analyzed by client size.")


DIAGNOSTIC: Checking GroupShuffleSplit Balance

Test Set Client Distribution:
Total clients in test: 11
Average samples per client: 354.3
Median samples per client: 29.0
Min samples per client: 1
Max samples per client: 1863

Client Size Breakdown:
  Clients with < 10 samples: 3
  Clients with 10-100 samples: 5
  Clients with 100-1000 samples: 1
  Clients with > 1000 samples: 2

DIAGNOSTIC: Client Size vs Performance

Performance by Client Size Group:
                  n_samples                   auc         accuracy        
                      count   sum    mean    mean     std     mean     std
size_group                                                                
Tiny (<10)                3     9     3.0  0.6111  0.1925   0.7500  0.2500
Small (10-100)            5   171    34.2  0.7137  0.1684   0.7702  0.0921
Medium (100-1000)         1   321   321.0  0.6926     NaN   0.6854     NaN
Large (>1000)             2  3396  1698.0  0.6623  0.0732   0.7360  0.1312
The GroupShuffleSp

/tmp/ipykernel_1126/3001299557.py:51: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(perf_df.groupby('size_group').agg({
